# Week 06 — Big Data and Efficient Processing

**Spatial Data Science with Python · MSc GIS / AGMS 1333**  
**Official Table 4 topic:** Machine Learning · Chunking, Big Data, efficiency

**Learning outcome.** Compare full-read and chunked table summaries.

**What you will implement:** Chunked CSV, sum/count aggregation, correctness assertion.  
**Mathematical model:** Pooled mean = Σ chunk sums / Σ valid counts.

> **Data provenance:** The Urban Teaching Grid is a **synthetic** 12 × 12 grid at 250-m cell spacing. Its temperatures and facilities are simulated; do not present its numerical outputs as field measurements. All distance calculations in the labs use the projected coordinates EPSG:32647 where relevant.

**How to run.** In Colab, upload this notebook from *File → Upload notebook*. Run each cell top-to-bottom. The setup cell below will ask for `DATA_ONLY.zip` only if the data folder is absent. In JupyterLab, extract the complete ZIP and open this notebook in `01_Core_Labs/`.


## 1. Prepare the environment and locate the data

The same setup works in JupyterLab and Google Colab. It selects an existing local data directory, or prompts for the packaged dataset in Colab. `assert` fails early if the archive is incomplete.

In [ ]:
from pathlib import Path
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local Jupyter: data/ or ../data. Colab: upload DATA_ONLY.zip once per runtime.
candidates = [Path('data'), Path('../data'), Path('/content/data')]
DATA = next((p for p in candidates if (p / 'cells.csv').is_file()), None)
if DATA is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Cannot find data/. Extract the course ZIP and keep the data folder with the notebooks.') from exc
    print('Select DATA_ONLY.zip from the downloaded course package:')
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
    if zip_name is None:
        raise FileNotFoundError('Please upload DATA_ONLY.zip and run this cell again.')
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        archive.extractall(Path.cwd())
    DATA = Path('data')
assert (DATA / 'metadata.json').is_file(), 'Data package incomplete.'
print('Dataset folder:', DATA.resolve())
print('Synthetic teaching data: not field observations.')


## 2. Key concepts and expected data

The analysis follows **input → cleaning → computation → validation → interpretation**. Always inspect observation keys, coordinate units, missing values, and the intended study period before fitting a model.

**Mathematical foundation:** Pooled mean = Σ chunk sums / Σ valid counts

**Method family:** Chunked CSV, sum/count aggregation, correctness assertion.

## 3. Main computation

Run this cell after the setup. The block calculates the principal statistics or fits the weekly model. The original course code is retained so results remain aligned with the slide examples.

In [ ]:
benchmark=DATA/'benchmark_repeated_rows.csv'
partial=[]
for chunk in pd.read_csv(benchmark,usecols=['date','lst_c'],chunksize=4000):
    partial.append(chunk.groupby('date').lst_c.agg(['sum','count']))
g=pd.concat(partial).groupby(level=0).sum()
streamed=g['sum']/g['count']
print(streamed)

### Reading the main code

**Stream.** `pd.read_csv(..., chunksize=4000)` reads at most 4,000 records at a time, selecting only needed columns.

- `pd.read_csv` — Reads CSV text into a pandas DataFrame. Inspect head() and column types after loading.
- `groupby` — Groups rows before counting or aggregating by sector, date, or other key.

**Verification questions:** Which variables contain the inputs? What new object does the cell create? What numerical result should be checked before continuing?

## 4. Map, plot, or evaluate the result

The final block turns results into a labelled graphic or output artifact. Keep units in axes/legends, and distinguish model predictions from observations.

In [ ]:
full=pd.read_csv(benchmark).groupby('date').lst_c.mean()
print('Max absolute difference:',(full-streamed).abs().max())
assert np.allclose(full,streamed)
plt.plot(full.values,label='Full read',marker='o')
plt.scatter(range(len(streamed)),streamed.values,marker='x',s=100,label='Chunked')
plt.legend();plt.title('The statistic is preserved');plt.show()

### Reading the output code

**Aggregate correctly.** Store `sum` and valid `count` for every chunk/date; then merge sums and counts by date.

- `pd.read_csv` — Reads CSV text into a pandas DataFrame. Inspect head() and column types after loading.
- `assert` — Checks a claimed condition and stops if it is false.
- `plt.` — Creates a figure; title, axes, and legend are part of the analytical interpretation.

**Verify.** `np.allclose` confirms the streamed result matches a full-read reference to numerical tolerance. Repeated rows are a computational stress test, not new independent field observations.

## 5. Critical interpretation

### Interpret your results

Complete the following short report **after** running the notebook:

1. **Research question:** What geographic question does this notebook address?
2. **Observation unit and period:** What does one row/pixel represent? What is its date and CRS?
3. **Method:** Why is this method suitable, and what assumptions does it make?
4. **Results:** Report two numerical outputs or figure patterns with units.
5. **Validity:** State one data limitation and one model/evaluation limitation.
6. **Extension:** Modify one parameter, compare results, and describe what changed.


## 6. Questions and independent practice

1. Change chunk size while preserving the pooled mean. Compare numerical agreement and measured runtime.
2. What would happen if the spatial unit, study date, or CRS changed?
3. Is this analysis descriptive, inferential, predictive, or a decision-support example? Justify your answer.
4. Which assumption should be tested before use in a real project?

**Next step:** Open the matching advanced notebook in `02_Advanced_Experiments/` for four short method variations.